# GLiNER 2.5 Decide — LoRA fine-tune scaffold (Colab, T4 16 GB)

**Task.** Business entity resolution: given a source-1 record and a candidate from source-2/3, decide same-business vs. different-business.

**Approach.** GLiNER 2.5 Decide ships with a proprietary `gliner2.AutoExtractor` head. Rather than hacking into that head, this notebook does LoRA on the underlying **mDeBERTa-v3-base** encoder as a standard binary sequence classifier over pair text. That gives us a stock HuggingFace + PEFT training path. Section 8 has notes on the alternate `gliner2`-native path if that's preferred.

**Runtime.** Assumes a free-tier T4 (16 GB VRAM). Defaults are conservative: fp16, small batch, gradient accumulation. Runs top-to-bottom without edits after Section 1 is filled in.

**Data location.** Expects the 7 TSVs under `MyDrive/amazon-entity-resolution/dataset/{train,test}/` — see the setup instructions from Claude for how to place them.

## 1. Environment

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q --upgrade pip
%pip install -q \
    "transformers>=4.44,<5" \
    "peft>=0.11" \
    "accelerate>=0.33" \
    "datasets>=2.20" \
    "evaluate>=0.4" \
    "scikit-learn" \
    "sentencepiece" \
    "huggingface_hub>=0.24" \
    "bitsandbytes>=0.43" \
    "gliner2"

## 2. Secrets & Drive

Add your HuggingFace token to Colab secrets: sidebar → 🔑 → New secret, name **`HF_TOKEN`**, paste the token, toggle notebook access.

In [ ]:
import os
from google.colab import userdata, drive

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/amazon-entity-resolution")
DATASET_DIR = DRIVE_ROOT / "dataset"
TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CACHE_DIR = Path("/content/models/GLiNER2.5-multi-Decide")
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

for p in [TRAIN_DIR / "train_ground_truth.tsv",
         TRAIN_DIR / "train_source1.tsv",
         TRAIN_DIR / "train_source2.tsv",
         TRAIN_DIR / "train_source3.tsv",
         TEST_DIR / "test_source1.tsv",
         TEST_DIR / "test_source2.tsv",
         TEST_DIR / "test_source3.tsv"]:
    print("OK " if p.exists() else "MISSING ", p)

## 3. Download GLiNER 2.5 Decide to local SSD

1.1 GB. Kept on `/content` (local SSD) — faster than reading weights over the Drive mount every run.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="fastino/GLiNER2.5-multi-Decide",
    local_dir=str(MODEL_CACHE_DIR),
    token=os.environ["HF_TOKEN"],
)
!ls -lh {MODEL_CACHE_DIR}

## 4. Peek at the data

In [ ]:
import pandas as pd

src_dtype = {"entity_id": "string", "business_name": "string",
             "business_address": "string", "country": "string"}

src1 = pd.read_csv(TRAIN_DIR / "train_source1.tsv", sep="\t", dtype=src_dtype)
src2 = pd.read_csv(TRAIN_DIR / "train_source2.tsv", sep="\t", dtype=src_dtype)
src3 = pd.read_csv(TRAIN_DIR / "train_source3.tsv", sep="\t", dtype=src_dtype)
gt = pd.read_csv(TRAIN_DIR / "train_ground_truth.tsv", sep="\t", dtype="string")

print(f"source1: {len(src1):,}   source2: {len(src2):,}   source3: {len(src3):,}")
print(f"ground truth rows: {len(gt):,}")
gt.head(3)

## 5. Build labeled pairs (positives + sampled negatives)

Mirrors `code/business_entity_resolution/src/build_labeled_pairs.py`. Positives from ground truth; 3 random source-2/3 negatives per positive. For a first fine-tune run, we subsample to a manageable size.

In [ ]:
import random

NEG_PER_POS = 3
SEED = 42
MAX_S1 = 20_000        # sub-sample size for iteration; raise to None for full data

rng = random.Random(SEED)

s1_lookup = src1.set_index("entity_id")
s2_lookup = src2.set_index("entity_id")
s3_lookup = src3.set_index("entity_id")
s2_ids = src2["entity_id"].tolist()
s3_ids = src3["entity_id"].tolist()
pool = s2_ids + s3_ids

def record_text(name, addr):
    return f"{name} | {addr or 'no address'}"

def lookup(cand_id):
    if cand_id.startswith("S2-"):
        row = s2_lookup.loc[cand_id] if cand_id in s2_lookup.index else None
    elif cand_id.startswith("S3-"):
        row = s3_lookup.loc[cand_id] if cand_id in s3_lookup.index else None
    else:
        return None
    if row is None: return None
    return row["business_name"], row["business_address"]

pairs = []  # (s1_text, cand_text, label)
gt_iter = gt.itertuples(index=False)
if MAX_S1 is not None:
    gt_iter = list(gt_iter)[:MAX_S1]

for row in gt_iter:
    s1_id = row.source1_entity_id
    if s1_id not in s1_lookup.index: continue
    s1_row = s1_lookup.loc[s1_id]
    s1_txt = record_text(s1_row["business_name"], s1_row["business_address"])
    matches = [m for m in (row.matched_entity_ids or "").split(",") if m]
    positives = set(matches)

    for m in matches:
        c = lookup(m)
        if c is None: continue
        pairs.append((s1_txt, record_text(*c), 1))

    n_neg = NEG_PER_POS * max(len(matches), 1)
    for _ in range(n_neg):
        for _try in range(20):
            cand_id = rng.choice(pool)
            if cand_id in positives: continue
            c = lookup(cand_id)
            if c is None: continue
            pairs.append((s1_txt, record_text(*c), 0))
            break

print(f"pairs: {len(pairs):,}  (positives: {sum(1 for _,_,l in pairs if l==1):,})")

## 6. Base model + LoRA config

We initialize a fresh sequence-classification head on top of `microsoft/mdeberta-v3-base` (the same encoder GLiNER 2.5 was built from — see `config.json: model_name`). LoRA targets the query/value projections of every attention layer.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

BASE_MODEL = "microsoft/mdeberta-v3-base"
MAX_LEN = 192

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    torch_dtype=torch.float16,
)

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["query_proj", "value_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## 7. Tokenize & train

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

train_pairs, eval_pairs = train_test_split(pairs, test_size=0.1, random_state=SEED,
                                           stratify=[l for _, _, l in pairs])

def to_ds(rows):
    return Dataset.from_dict({
        "text_a": [r[0] for r in rows],
        "text_b": [r[1] for r in rows],
        "label": [r[2] for r in rows],
    })

train_ds = to_ds(train_pairs)
eval_ds = to_ds(eval_pairs)

def tok(batch):
    return tokenizer(batch["text_a"], batch["text_b"],
                     truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tok, batched=True, remove_columns=["text_a", "text_b"])
eval_ds = eval_ds.map(tok, batched=True, remove_columns=["text_a", "text_b"])

In [ ]:
import numpy as np
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer
from sklearn.metrics import f1_score, precision_score, recall_score

collator = DataCollatorWithPadding(tokenizer=tokenizer)

def metrics(pred):
    y_true = pred.label_ids
    y_pred = np.argmax(pred.predictions, axis=-1)
    return {
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
    }

args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR / "lora-mdeberta-decide"),
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.06,
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=metrics,
)
trainer.train()

In [ ]:
adapter_out = CHECKPOINT_DIR / "lora-mdeberta-decide-final"
model.save_pretrained(str(adapter_out))
tokenizer.save_pretrained(str(adapter_out))
print("Saved adapter to", adapter_out)

## 8. Alternative path — LoRA on the actual GLiNER 2.5 Decide head

The section above trains a fresh classifier on `mdeberta-v3-base`. If instead you want to keep GLiNER Decide's `BoundaryExtractor` head (bucket-mode scoring is closer to how `gliner_decide.py` uses the model at inference), the two options are:

1. **Fastino hosted training** — the model's `SKILL.md` documents `POST /v1/training-jobs` with LoRA support. Requires a Fastino API key; happens off-Colab.
2. **Roll your own** — subclass `gliner2.AutoExtractor`, unfreeze only the encoder's attention projections, wrap them with `peft.LoraLayer`, and drive the existing bucket loss. Non-trivial because the head is proprietary; expect debugging.

Ping to expand this into runnable code if that's the direction.

## 9. What to try next

- **Real negatives** > random negatives: swap the negative sampler for the shortlisted-candidate output of `blocking.py` + `fuzzy.py`. Random negatives are almost trivially easy; hard negatives are where the signal lives.
- **Class balance**: 3:1 negatives is arbitrary; sweep.
- **Address normalization**: `normalize.py` in the repo. Applying it before tokenization usually helps mDeBERTa on this task.
- **Multilingual eval**: the test set spans Hindi/CJK; check per-language F1, not just aggregate.